In [1]:
import json
from collections import Counter

def explore_kc_routes(input_file, output_txt):
    print(f"Đang đọc dữ liệu từ {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    all_routes = []
    
    # Duyệt qua từng câu hỏi để lấy kc_routes
    for key, item in data.items():
        kc_routes = item.get("kc_routes", [])
        # kc_routes thường là một mảng (list), ta duyệt qua từng phần tử trong mảng
        if isinstance(kc_routes, list):
            for route in kc_routes:
                all_routes.append(route.strip())
        elif isinstance(kc_routes, str):
            all_routes.append(kc_routes.strip())
            
    # Đếm tần suất xuất hiện của từng tuyến kiến thức
    route_counter = Counter(all_routes)
    
    # Ghi kết quả ra file Text, sắp xếp từ phổ biến nhất đến ít nhất
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write(f"TỔNG SỐ TUYẾN KIẾN THỨC DUY NHẤT: {len(route_counter)}\n")
        f.write("="*50 + "\n\n")
        for route, count in route_counter.most_common():
            f.write(f"[{count} lần] {route}\n")
            
    print(f"Đã lưu thống kê vào file: {output_txt}")

# Chạy thử
explore_kc_routes("D:\\IntelligentTesting\\IntelligentTesting\\XES3G5M\\XES3G5M\\metadata\\question_full.json", "thong_ke_kc_routes.txt")

Đang đọc dữ liệu từ D:\IntelligentTesting\IntelligentTesting\XES3G5M\XES3G5M\metadata\question_full.json...
Đã lưu thống kê vào file: thong_ke_kc_routes.txt


In [2]:
import json

def get_unique_skills(json_filepath):
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    unique_full_routes = set()
    unique_core_skills = set()

    for item in data.values():
        kc_routes = item.get("kc_routes", [])
        
        for route in kc_routes:
            # 1. Lưu lại toàn bộ chuỗi dài nguyên bản
            unique_full_routes.add(route.strip())
            
            # 2. Cắt nhỏ chuỗi để lấy các "kỹ năng lõi" (tách bởi "----")
            nodes = [node.strip() for node in route.split("----") if node.strip()]
            unique_core_skills.update(nodes)

    # In kết quả ra file Text để bạn dễ phân tích
    with open("unique_skills.txt", "w", encoding="utf-8") as f:
        f.write(f"=== TỔNG SỐ CHUỖI NHÃN DUY NHẤT: {len(unique_full_routes)} ===\n")
        f.write("\n".join(sorted(unique_full_routes)))
        
        f.write("\n\n" + "="*50 + "\n\n")
        
        f.write(f"=== TỔNG SỐ KỸ NĂNG LÕI DUY NHẤT (Đã cắt nhỏ): {len(unique_core_skills)} ===\n")
        f.write("\n".join(sorted(unique_core_skills)))
        
    print(f"Đã trích xuất xong! Vui lòng mở file 'unique_skills.txt' để xem.")

# Chạy hàm
get_unique_skills("D:\\IntelligentTesting\\IntelligentTesting\\XES3G5M\\XES3G5M\\metadata\\question_full.json")

Đã trích xuất xong! Vui lòng mở file 'unique_skills.txt' để xem.


In [3]:
def extract_vector_q(kc_routes_list):
    """
    Chuyển đổi danh sách kc_routes thành Vecto nhị phân độ dài K (VD: K=6)
    """
    # 1. Định nghĩa TỪ KHÓA ĐẠI DIỆN cho từng chiều của Vector (Dựa vào kết quả Bước 1)
    # Lưu ý: Viết chữ thường (lowercase) để dễ so sánh
    skill_mapping = {
        "Q1_SoHoc_PhanSo": ["số tự nhiên", "phân số", "số thập phân", "tính toán"],
        "Q2_TiSo_PhanTram": ["tỉ số", "phần trăm", "lãi", "lỗ", "nồng độ"],
        "Q3_ChuyenDong": ["chuyển động", "vận tốc", "quãng đường", "ngược chiều", "cùng chiều"],
        "Q4_HinhHoc": ["hình học", "chu vi", "diện tích", "thể tích", "hình tròn", "lập phương"],
        "Q5_DoLuong": ["đo lường", "đổi đơn vị", "khối lượng", "thời gian"],
        "Q6_ToanTuDuy": ["tư duy", "nâng cao", "phần đếm", "nguyên lý nhân", "logic"]
    }
    
    # 2. Khởi tạo Vecto Q toàn số 0 (Độ dài K=6)
    # Kết quả sẽ có dạng dictionary để bạn dễ đưa vào ma trận sau này
    vector_q = {key: 0 for key in skill_mapping.keys()}
    
    # Ép kiểu dữ liệu đầu vào thành chuỗi viết thường để dễ tìm kiếm
    routes_text = str(kc_routes_list).lower()
    
    # 3. Quét từ khóa để BẬT CỜ (Chuyển 0 thành 1)
    for q_key, keywords in skill_mapping.items():
        # Nếu có BẤT KỲ từ khóa nào trong keywords xuất hiện trong routes_text
        if any(keyword in routes_text for keyword in keywords):
            vector_q[q_key] = 1  # Bật cờ kỹ năng này lên 1
            
    return vector_q

# ==========================================
# TEST THỬ VỚI CÂU HỎI SỐ "0" CỦA BẠN
# ==========================================
sample_kc_routes = [
    "Toán nâng cao----Phần đếm----Nguyên lý cộng và nhân----Nguyên lý nhân----Các loại khác trong nguyên lý nhân"
]

ket_qua_vecto = extract_vector_q(sample_kc_routes)
print("Kết quả Vecto Q:")
for k, v in ket_qua_vecto.items():
    print(f"- {k}: {v}")

Kết quả Vecto Q:
- Q1_SoHoc_PhanSo: 0
- Q2_TiSo_PhanTram: 0
- Q3_ChuyenDong: 0
- Q4_HinhHoc: 0
- Q5_DoLuong: 0
- Q6_ToanTuDuy: 1
